## Limpieza y Preparación de Datos

Antes de proceder con el análisis y la simulación, es crucial limpiar y estandarizar los datos de las tres fuentes principales:

1.  **Datos Históricos de Partidos (`datos_historicos.csv`):**
    *   Conversión de la columna 'Fecha' a formato `datetime`.
    *   Normalización de los nombres de los equipos (`Equipo_Local`, `Equipo_Visitante`) para asegurar consistencia en los cruces de datos (ej. 'EEUU' a 'Estados Unidos').
    *   Conversión de columnas numéricas que se cargaron como objetos.
    *   Creación de una columna `Resultado_Num` para facilitar el análisis del resultado (1 para victoria local, 0 para empate, -1 para victoria visitante).
    *   Eliminación de filas incompletas o sin información esencial.

2.  **Ranking FIFA (`ranking_fifa.csv`):**
    *   Conversión de la columna 'Fecha' a formato `datetime`.
    *   Normalización de nombres de países para coincidir con los otros datasets.
    *   Filtrado para obtener el ranking más reciente de cada país, eliminando duplicados por país y fecha.

3.  **Datos de Transfermarkt (`transfermarkt.csv`):**
    *   Normalización de los nombres de los equipos.
    *   Ordenamiento de los equipos por valor de mercado para una rápida visualización de los top equipos.

Estas operaciones aseguran que los datos estén en un formato adecuado y consistente para la siguiente fase de integración y análisis.

In [ ]:
import pandas as pd
import numpy as np

# Cargar
df_h = pd.read_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/datos_historicos.csv',
                   encoding='utf-8', low_memory=False)

print("Dimensiones originales:", df_h.shape)

# ====================== LIMPIEZA ======================
# 1. Convertir Fecha a datetime
df_h['Fecha'] = pd.to_datetime(df_h['Fecha'], errors='coerce')

# 2. Normalizar nombres de equipos (importante para cruces)
def normalizar_equipo(x):
    if pd.isna(x): return x
    x = str(x).strip()
    # Correcciones comunes
    replacements = {
        'EEUU': 'Estados Unidos',
        'Chequia': 'República Checa',
        'Corea del Sur': 'República de Corea',
        'Arabia Saudita': 'Arabia Saudí',
        'Irán': 'RI de Irán'
    }
    for old, new in replacements.items():
        if old in x:
            x = x.replace(old, new)
    return x

for col in ['Equipo_Local', 'Equipo_Visitante']:
    df_h[col] = df_h[col].apply(normalizar_equipo)

# 3. Convertir columnas numéricas
numeric_cols = df_h.select_dtypes(include=['object']).columns
for col in numeric_cols:
    if col not in ['Fecha', 'Equipo_Local', 'Equipo_Visitante', 'Resultado_1X2']:
        df_h[col] = pd.to_numeric(df_h[col], errors='coerce')

# 4. Crear resultado numérico (opcional pero muy útil)
def resultado_num(r):
    if r == '1': return 1
    elif r == 'X': return 0
    elif r == '2': return -1
    else: return np.nan

df_h['Resultado_Num'] = df_h['Resultado_1X2'].apply(resultado_num)

# 5. Eliminar filas sin fecha o sin equipos
df_h = df_h.dropna(subset=['Fecha', 'Equipo_Local', 'Equipo_Visitante'])

print("Dimensiones después de limpieza:", df_h.shape)
print("\nColumnas disponibles:")
print(df_h.columns.tolist())

# Guardar versión limpia
df_h.to_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/datos_historicos_limpio.csv', index=False)
print("✅ Archivo guardado: datos_historicos_limpio.csv")

Dimensiones originales: (1396, 48)
Dimensiones después de limpieza: (1396, 49)

Columnas disponibles:
['Fecha', 'Equipo_Local', 'Equipo_Visitante', 'Goles_Local', 'Goles_Visitante', 'Peso_Local', 'avg_Goles_esperados_(xG)_total_Local', 'avg_Tarjetas_amarillas_total_Local', 'avg_Faltas_total_Local', 'avg_Remates_a_puerta_3_Local', 'avg_Córneres_3_Local', 'avg_Tarjetas_amarillas_3_Local', 'avg_Faltas_3_Local', 'avg_Paradas_3_Local', 'avg_Pases_Pct_3_Local', 'avg_Pases_Exitosos_3_Local', 'Peso_Visitante', 'Resultado_1X2', 'avg_Goles_esperados_(xG)_total_Visitante', 'avg_Tarjetas_amarillas_total_Visitante', 'avg_Faltas_total_Visitante', 'avg_Pases_Pct_total_Visitante', 'avg_xG_Share_total_Visitante', 'avg_Goles_esperados_(xG)_3_Visitante', 'avg_Remates_a_puerta_3_Visitante', 'avg_Córneres_3_Visitante', 'avg_Tarjetas_amarillas_3_Visitante', 'avg_Faltas_3_Visitante', 'avg_Paradas_3_Visitante', 'avg_Pases_Exitosos_3_Visitante', 'avg_Goles_3_Visitante', 'avg_Ptos_3_Visitante', 'avg_xG_Share_3_

In [ ]:
import pandas as pd
from datetime import datetime

df_r = pd.read_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/ranking_fifa.csv',
                   encoding='utf-8')

print("Original:", df_r.shape)

# Limpiar
df_r['Fecha'] = pd.to_datetime(df_r['Fecha'], format='%d-%m-%Y', errors='coerce')
df_r['País'] = df_r['País'].str.strip()

# Normalizar algunos nombres
df_r['País'] = df_r['País'].replace({
    'EEUU': 'Estados Unidos',
    'Chequia': 'República Checa',
    'Corea del Sur': 'República de Corea',
    'Arabia Saudita': 'Arabia Saudí'
})

# Ordenar por fecha descendente
df_r = df_r.sort_values(['País', 'Fecha'], ascending=[True, False])

# Quedarnos con el ranking más reciente por país
df_r_latest = df_r.drop_duplicates(subset='País', keep='first')

print("Ranking más reciente por país:", df_r_latest.shape)
print(df_r_latest.head())

df_r_latest.to_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/ranking_fifa_limpio.csv', index=False)
print("✅ Archivo guardado: ranking_fifa_limpio.csv")

Original: (6931, 3)
Ranking más reciente por país: (217, 3)
                  País  Puntuación      Fecha
6787  ARY de Macedonia     1343.00 2018-12-20
168         Afganistán      972.75 2026-04-01
63             Albania     1388.06 2026-04-01
9             Alemania     1730.37 2026-04-01
172            Andorra      945.34 2026-04-01
✅ Archivo guardado: ranking_fifa_limpio.csv


In [ ]:
import pandas as pd

df_t = pd.read_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/transfermarkt.csv',
                   encoding='utf-8')

print("Original:", df_t.shape)

# Limpieza básica
df_t['Equipo'] = df_t['Equipo'].str.strip()

# Normalizar nombres
df_t['Equipo'] = df_t['Equipo'].replace({
    'Estados Unidos': 'EEUU',
    'Chequia': 'República Checa',
    'Arabia Saudita': 'Arabia Saudí',
    'Corea del Sur': 'República de Corea'
})

df_t = df_t.sort_values('Valor_Mercado_Millones_Eur', ascending=False)

print("Top 10 valores de mercado:")
print(df_t.head(10)[['Equipo', 'Valor_Mercado_Millones_Eur']])

df_t.to_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/transfermarkt_limpio.csv', index=False)
print("✅ Archivo guardado: transfermarkt_limpio.csv")

Original: (217, 2)
Top 10 valores de mercado:
         Equipo  Valor_Mercado_Millones_Eur
0       Francia                       58.58
1    Inglaterra                       52.24
2        España                       47.03
3      Portugal                       38.67
4      Alemania                       36.42
5        Brasil                       35.70
6     Argentina                       31.06
7  Países Bajos                       29.01
8       Noruega                       22.69
9       Bélgica                       21.06
✅ Archivo guardado: transfermarkt_limpio.csv
